# Aula 1 - Problema e primeiro pipeline
## Do dado bruto a primeira decisao computacional

**Objetivo:** Configurar o ambiente e transformar uma imagem da camera em uma primeira decisao computacional.

**Atividade do dia:** Capturar video da camera, detectar uma cor ou marcador visual e produzir uma classificacao generica de posicao. A aplicacao ao projeto final fica para depois da escolha do grupo.

---

### Como executar

Cada celula de camera tem um tempo limite (`duracao_s`). A celula para sozinha quando o tempo acaba,
ou antes se voce pressionar **Q** na janela de video. Assim o notebook nunca trava.

Para rodar sem abrir janelas (verificacao automatica), defina a variavel de ambiente `LCML_AUTO=1`
antes de abrir o Jupyter.

## Bibliotecas que vamos usar

Antes de programar, vamos entender quais bibliotecas Python usaremos neste projeto e para que cada uma serve.

### Bibliotecas de hoje

| Biblioteca | Para que serve |
|-----------|---------------|
| **NumPy** | Manipular arrays (matrizes) de numeros. Uma imagem digital nada mais e que uma matriz NumPy. Cada pixel e um numero. |
| **OpenCV** (`cv2`) | Biblioteca de visao computacional. Abre a camera, le imagens, converte espacos de cor (BGR, HSV, cinza), detecta contornos e desenha na tela. |
| **matplotlib** (`plt`) | Gera graficos e mostra imagens dentro do notebook. |

### Bibliotecas das proximas aulas (ja instalamos hoje)

| Biblioteca | Para que serve | Aula |
|-----------|---------------|------|
| **pandas** | Manipular dados em tabela (DataFrame). Ler e escrever CSV com metadados e resultados. | 2, 5, 8 |
| **scikit-learn** | Aprendizado de maquina classico. Treina Decision Tree, Random Forest, SVM e k-NN. Calcula acuracia, precisao, revocacao. | 3, 5 |
| **seaborn** | Graficos estatisticos. Usaremos para plotar a matriz de confusao de forma colorida. | 3, 5 |
| **joblib** | Salvar e carregar modelos treinados. Permite reusar o modelo na inferencia sem retreinar. | 3, 6 |
| **PyTorch** (`torch`) | Framework de deep learning. Define e treina redes neurais. Suporta GPU para acelerar o treino. | 4 |
| **torchvision** | Modelos de visao pre-treinados (MobileNetV2, ResNet) e ferramentas de imagem para PyTorch. | 4 |
| **pyserial** (`serial`) | Comunicacao serial pela porta USB com Arduino ou ESP32. Sera usada quando o projeto escolhido precisar de hardware. | 6, 7 |

### Analogia

Pense no projeto como uma cozinha:

- **NumPy** = ingredientes basicos (tudo comeca aqui)
- **OpenCV** = faca e tabua (processa as imagens)
- **scikit-learn** = forno simples (treina o baseline)
- **PyTorch** = forno profissional (treina redes neurais)
- **matplotlib/seaborn** = os pratos (apresenta o resultado)
- **pyserial** = o entregador (leva o comando ao Arduino)
- **pandas** = caderno de receitas (organiza os dados)

In [ ]:
# Importar todas as bibliotecas do projeto e verificar versoes

# --- Bibliotecas de hoje ---
import numpy as np
print(f'NumPy {np.__version__} - arrays e matrizes numericas')

import cv2
print(f'OpenCV {cv2.__version__} - visao computacional e camera')

import matplotlib
print(f'matplotlib {matplotlib.__version__} - graficos e visualizacao')

import matplotlib.pyplot as plt

# --- Bibliotecas das proximas aulas ---
import pandas as pd
print(f'pandas {pd.__version__} - manipulacao de dados em tabela')

import sklearn
print(f'scikit-learn {sklearn.__version__} - aprendizado de maquina classico')

import seaborn as sns
print(f'seaborn {sns.__version__} - graficos estatisticos')

import joblib
print(f'joblib {joblib.__version__} - salvar/carregar modelos')

import torch
print(f'PyTorch {torch.__version__} - deep learning e redes neurais')
if torch.cuda.is_available():
    print(f'  GPU disponivel: {torch.cuda.get_device_name(0)}')
else:
    print(f'  GPU nao disponivel - usaremos CPU')

import torchvision
print(f'torchvision {torchvision.__version__} - modelos de visao pre-treinados')

import serial
print(f'pyserial {serial.__version__} - comunicacao com Arduino')

print()
print('Todas as bibliotecas carregadas com sucesso!')

## Passo 1: Verificar o ambiente

In [ ]:
import sys
print(f"Python: {sys.version}")

import numpy as np
import cv2
print(f"NumPy: {np.__version__}")
print(f"OpenCV: {cv2.__version__}")
print("Ambiente OK!")

## Passo 1b: Kit da aula

Esta celula define as funcoes que vamos reutilizar nos proximos passos. Assim cada passo fica curto
e a regra de decisao fica escrita em um lugar so.

- `abrir_camera()` - abre a webcam (no Windows usamos o backend DSHOW, que inicia mais rapido).
- `rodar_camera(processar, duracao_s)` - le quadros em loop, aplica a sua funcao em cada quadro e
  para por tempo ou pela tecla **Q**. Devolve o ultimo quadro processado.
- `mostrar(img)` - exibe uma imagem dentro do notebook com matplotlib (converte BGR para RGB).

In [ ]:
import os, time, pathlib
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Modo automatico: nao abre janelas (util para executar o notebook sem interacao)
AUTO = os.environ.get("LCML_AUTO") == "1"

PASTA_ENTREGAS = pathlib.Path("entregas")
PASTA_ENTREGAS.mkdir(exist_ok=True)


def abrir_camera(indice=0):
    """Abre a webcam e devolve o objeto VideoCapture."""
    backend = cv2.CAP_DSHOW if os.name == "nt" else cv2.CAP_ANY
    cap = cv2.VideoCapture(indice, backend)
    if not cap.isOpened():
        raise RuntimeError(
            f"Nao foi possivel abrir a camera {indice}. "
            "Verifique se outro programa esta usando a webcam e se o Windows "
            "permite acesso a camera (Configuracoes > Privacidade > Camera)."
        )
    return cap


def rodar_camera(processar, duracao_s=10, titulo="Camera", indice=0, mostrar_janela=None):
    """Le quadros da camera e aplica `processar(frame)` em cada um.

    `processar` recebe o quadro BGR e devolve o quadro que deve ser exibido
    (ou uma tupla de quadros, que serao mostrados em janelas separadas).
    O loop para quando `duracao_s` acaba ou quando voce pressiona Q.
    Devolve o ultimo resultado de `processar`.
    """
    if mostrar_janela is None:
        mostrar_janela = not AUTO

    cap = abrir_camera(indice)
    t0 = time.time()
    n = 0
    ultimo = None
    try:
        while time.time() - t0 < duracao_s:
            ret, frame = cap.read()
            if not ret:
                print("ERRO: nao foi possivel ler o quadro.")
                break
            n += 1
            ultimo = processar(frame)
            if mostrar_janela:
                saidas = ultimo if isinstance(ultimo, tuple) else (ultimo,)
                for i, img in enumerate(saidas):
                    nome = titulo if i == 0 else f"{titulo} [{i}]"
                    cv2.imshow(nome, img)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
    finally:
        cap.release()
        if mostrar_janela:
            cv2.destroyAllWindows()
            cv2.waitKey(1)

    decorrido = time.time() - t0
    print(f"{n} quadros em {decorrido:.1f}s ({n / max(decorrido, 1e-6):.1f} FPS)")
    return ultimo


def mostrar(img, titulo="", cmap=None, tamanho=(9, 5)):
    """Mostra uma imagem dentro do notebook (converte BGR->RGB quando for colorida)."""
    plt.figure(figsize=tamanho)
    if img.ndim == 2:
        plt.imshow(img, cmap=cmap or "gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(titulo)
    plt.axis("off")
    plt.show()


print("Kit carregado. Modo automatico (sem janelas):", AUTO)

## Passo 2: Testar a camera

A celula abaixo abre a webcam por alguns segundos e mostra o video ao vivo.
Pressione **Q** na janela para sair antes do tempo.

In [ ]:
ultimo_frame = rodar_camera(lambda frame: frame, duracao_s=8, titulo="Camera")

print("Camera fechada.")
mostrar(ultimo_frame, "Passo 2 - ultimo quadro capturado")

## Passo 3: Converter para HSV

Uma imagem colorida chega do OpenCV em **BGR**. Para detectar cor, o espaco **HSV** e melhor:

- **H** (matiz) = qual e a cor (0-179 no OpenCV)
- **S** (saturacao) = quao intensa ela e
- **V** (valor) = quao clara ela e

Separar a cor (H) do brilho (V) e o que faz a deteccao continuar funcionando quando a iluminacao muda.

In [ ]:
def ver_espacos_de_cor(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return frame, hsv, cinza


frame, hsv, cinza = rodar_camera(ver_espacos_de_cor, duracao_s=6, titulo="Espacos de cor")

h, s, v = cv2.split(hsv)
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, img, nome in zip(
    axes.ravel(),
    [cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), hsv, cinza, h, s, v],
    ["Original (RGB)", "HSV", "Cinza", "H - matiz", "S - saturacao", "V - valor"],
):
    ax.imshow(img, cmap=None if img.ndim == 3 else "gray")
    ax.set_title(nome)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Passo 4: Detectar uma cor

Agora limiarizamos no espaco HSV com `cv2.inRange` e procuramos contornos na mascara.

O intervalo padrao e o **verde**. Se voce nao tiver um objeto verde a mao, troque `COR_ALVO`
para `"vermelho"`, `"azul"`, `"amarelo"` ou `"laranja"` na celula abaixo.

In [ ]:
# Intervalos HSV por cor (OpenCV: H de 0 a 179)
FAIXAS = {
    "verde":    [(np.array([30,  30,  25]), np.array([95, 255, 255]))],  # calibrado no coco verde (casca escura)
    "azul":     [(np.array([95,  80,  50]), np.array([130, 255, 255]))],
    "amarelo":  [(np.array([20, 100, 100]), np.array([33, 255, 255]))],
    "laranja":  [(np.array([8,  120, 120]), np.array([20, 255, 255]))],
    # o vermelho fica nas duas pontas do circulo de matiz, por isso sao duas faixas
    "vermelho": [(np.array([0,  120,  80]), np.array([8,  255, 255])),
                 (np.array([170, 120, 80]), np.array([179, 255, 255]))],
}

COR_ALVO = "verde"
AREA_MINIMA = 500   # ignora manchas menores que isso (ruido)


def mascara_da_cor(frame, cor=None):
    """Devolve a mascara binaria dos pixels que pertencem a cor-alvo."""
    cor = cor or COR_ALVO
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mascara = None
    for baixo, alto in FAIXAS[cor]:
        m = cv2.inRange(hsv, baixo, alto)
        mascara = m if mascara is None else cv2.bitwise_or(mascara, m)
    # limpeza: remove pontinhos soltos e fecha buracos
    nucleo = np.ones((5, 5), np.uint8)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, nucleo)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, nucleo)
    return mascara


def maior_objeto(mascara, area_minima=None):
    """Acha o maior contorno da mascara. Devolve (contorno, area, centro)."""
    area_minima = AREA_MINIMA if area_minima is None else area_minima
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos:
        return None, 0.0, None
    maior = max(contornos, key=cv2.contourArea)
    area = cv2.contourArea(maior)
    if area < area_minima:
        return None, area, None
    M = cv2.moments(maior)
    if M["m00"] == 0:
        return None, area, None
    centro = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"]))
    return maior, area, centro


def detectar(frame):
    mascara = mascara_da_cor(frame)
    contorno, area, centro = maior_objeto(mascara)
    saida = frame.copy()
    if contorno is not None:
        cv2.drawContours(saida, [contorno], -1, (0, 255, 0), 2)
        cv2.circle(saida, centro, 20, (0, 255, 0), 3)
        cv2.putText(saida, f"Area: {area:.0f}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    else:
        cv2.putText(saida, f"procurando {COR_ALVO}...", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    return saida, cv2.cvtColor(mascara, cv2.COLOR_GRAY2BGR)


print(f"Mostre um objeto {COR_ALVO} para a camera.")
deteccao, mascara_vis = rodar_camera(detectar, duracao_s=12, titulo="Deteccao")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
a1.imshow(cv2.cvtColor(deteccao, cv2.COLOR_BGR2RGB)); a1.set_title("Deteccao"); a1.axis("off")
a2.imshow(mascara_vis[:, :, 0], cmap="gray");         a2.set_title("Mascara");   a2.axis("off")
plt.tight_layout()
plt.show()

## Passo 5: Produzir uma classificacao generica

Com o centro do objeto em maos, aplicamos a regra de decisao. A tela e dividida em tres zonas:
uma faixa central de tolerancia (`MARGEM`) e as duas laterais.

Este e o pipeline completo em miniatura:
**dado bruto (pixels) -> caracteristica (centro do objeto) -> decisao (ESQUERDA/CENTRO/DIREITA)**.

In [ ]:
MARGEM = 80  # metade da largura da zona "CENTRO", em pixels

CORES = {
    "ESQUERDA":     (0, 200, 255),
    "DIREITA":      (255, 100, 0),
    "CENTRO":       (0, 255, 0),
    "SEM_DETECCAO": (0, 0, 255),
}


def classificar_posicao(frame, margem=None, cor=None):
    """Pipeline completo: quadro -> comando. Devolve (comando, centro, area)."""
    margem = MARGEM if margem is None else margem
    largura = frame.shape[1]
    mascara = mascara_da_cor(frame, cor)
    contorno, area, centro = maior_objeto(mascara)

    if centro is None:
        return "SEM_DETECCAO", None, area

    cx = centro[0]
    meio = largura // 2
    if cx < meio - margem:
        comando = "ESQUERDA"
    elif cx > meio + margem:
        comando = "DIREITA"
    else:
        comando = "CENTRO"
    return comando, centro, area


def desenhar_decisao(frame, comando, centro, area):
    """Desenha as zonas, o objeto e o comando no quadro."""
    saida = frame.copy()
    altura, largura = saida.shape[:2]
    meio = largura // 2
    cor = CORES[comando]

    # linhas que delimitam a zona central
    for x in (meio - MARGEM, meio + MARGEM):
        cv2.line(saida, (x, 0), (x, altura), (200, 200, 200), 1)

    if centro is not None:
        cv2.circle(saida, centro, 20, cor, 3)
        cv2.line(saida, (centro[0], 0), (centro[0], altura), cor, 1)
        cv2.putText(saida, f"x={centro[0]}  area={area:.0f}", (10, altura - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, cor, 2)

    cv2.putText(saida, comando, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, cor, 3)
    return saida


historico = []


def passo5(frame):
    comando, centro, area = classificar_posicao(frame)
    historico.append(comando)
    return desenhar_decisao(frame, comando, centro, area)


historico.clear()
print(f"Mova um objeto {COR_ALVO} pela tela: esquerda, centro e direita.")
resultado = rodar_camera(passo5, duracao_s=20, titulo="Classificacao visual - Aula 1")

mostrar(resultado, "Passo 5 - classificacao")

import collections
print("Comandos observados:", dict(collections.Counter(historico)))

## Passo 6: Validar a regra de decisao

Antes de dizer que "funciona", vamos provar. Criamos quadros sinteticos com um objeto na
esquerda, no centro e na direita e conferimos se o pipeline devolve o comando esperado.

Isso e um **teste automatizado**: nao depende da iluminacao nem de voce ter um objeto colorido
em maos, e vai continuar valendo quando o grupo adaptar o codigo ao projeto escolhido.

In [ ]:
def quadro_sintetico(x, cor_bgr=(0, 255, 0), largura=640, altura=480, lado=80):
    """Cria um quadro preto com um retangulo colorido centrado em x."""
    f = np.zeros((altura, largura, 3), np.uint8)
    cv2.rectangle(f, (x - lado // 2, altura // 2 - lado // 2),
                     (x + lado // 2, altura // 2 + lado // 2), cor_bgr, -1)
    return f


casos = [
    (quadro_sintetico(100), "ESQUERDA"),
    (quadro_sintetico(320), "CENTRO"),
    (quadro_sintetico(540), "DIREITA"),
    (np.zeros((480, 640, 3), np.uint8), "SEM_DETECCAO"),
]

falhas = 0
paineis = []
for quadro, esperado in casos:
    obtido, centro, area = classificar_posicao(quadro)
    ok = obtido == esperado
    falhas += not ok
    print(f"esperado={esperado:12s} obtido={obtido:12s} centro={str(centro):10s} "
          f"area={area:6.0f}  {'OK' if ok else 'FALHOU'}")
    paineis.append(desenhar_decisao(quadro, obtido, centro, area))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, img, (_, esperado) in zip(axes, paineis, casos):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(esperado, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

assert falhas == 0, f"{falhas} caso(s) falharam"
print("\nTODOS OS CASOS PASSARAM - a regra de decisao esta correta.")

## Passo 7: Salvar a captura de tela da entrega

Grava na pasta `entregas/` as imagens que comprovam o experimento funcionando.

In [ ]:
from datetime import datetime

carimbo = datetime.now().strftime("%Y-%m-%d %H:%M")

painel = resultado.copy()
cv2.putText(painel, f"LCML Aula 1 - {carimbo}", (10, painel.shape[0] - 45),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

destino = PASTA_ENTREGAS / "aula01_classificacao.png"
cv2.imwrite(str(destino), painel)
print("Captura salva em:", destino.resolve())

grade = np.hstack([cv2.resize(p, (320, 240)) for p in paineis])
destino_testes = PASTA_ENTREGAS / "aula01_validacao.png"
cv2.imwrite(str(destino_testes), grade)
print("Validacao salva em:", destino_testes.resolve())

mostrar(painel, "Captura da entrega")

## Entrega da Aula 1

- [x] Notebook executado sem erros
- [x] Camera abre e detecta a cor
- [x] Classificacao (ESQUERDA/CENTRO/DIREITA) funciona
- [x] Captura de tela salva em `entregas/`
- [x] README.md atualizado com o problema escolhido pelo grupo

**Extensao:** Diagnosticar a demanda e definir o publico (3 h previstas).

### O que levamos para a Aula 2

A regra de hoje e **escrita a mao**: nos escolhemos o intervalo HSV e o valor da margem.
Ela quebra quando a iluminacao muda ou quando aparece outro objeto da mesma cor.
Na Aula 2 vamos coletar um conjunto de dados rotulado para que o **modelo aprenda** esses limites,
em vez de nos chutarmos os numeros.